# 01 - Set up the Finance Genie lakehouse

Download the Finance Genie sample data from the public
[graph-on-databricks](https://github.com/neo4j-partners/graph-on-databricks) repo and
build the five **Silver Delta tables** the Neo4j Virtual Graph reads. Run this once,
then build the Virtual Graph over these tables in Aura (see `virtual-graph.md`) and run
the demos with `uv run vg-demo`.

## What this notebook does
- Reads the committed CSVs through a `DATA_SOURCE` switch (GitHub or a local clone)
- Stages them into a Unity Catalog Volume
- Creates the five base tables with their Unity Catalog column comments
- Loads each table with typed columns and adds the informational foreign keys
- Verifies the row counts

## Prerequisites
- A Databricks workspace; run this notebook on a cluster or serverless with internet
  egress (the GitHub download needs it).
- Permission to create a schema, volume, and tables in the target catalog. The catalog
  itself must already exist (see Section 2).

Finance Genie is a synthetic dataset of bank accounts, merchants, and the transfers
between them, so it is safe to load into a demo workspace.

## Section 1: Configuration

Pick the target catalog / schema / volume and where to read the data from. The
defaults match the Finance Genie pipeline; point `CATALOG` at any catalog you can write
to. Whatever you choose here is what you enter when building the Virtual Graph in Aura
(`virtual-graph.md`).

In [ ]:
# ============================================
# CONFIGURATION
# ============================================

CATALOG = "graph-on-databricks"
SCHEMA  = "graph-enriched-schema"
VOLUME  = "graph-enriched-volume"

# Where to read the CSVs from:
#   "github" -> raw files from the public repo (default, zero setup)
#   "local"  -> a directory you have already populated with the Finance Genie CSVs
#               (e.g. a clone of graph-on-databricks/finance-genie/data)
DATA_SOURCE = "github"

GITHUB_DATA_BASE = (
    "https://raw.githubusercontent.com/neo4j-partners/"
    "graph-on-databricks/main/finance-genie/data"
)
LOCAL_DATA_DIR = "finance-genie/data"  # used only when DATA_SOURCE == "local"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
CSV_FILES = [
    "accounts.csv",
    "merchants.csv",
    "transactions.csv",
    "account_links.csv",
    "account_labels.csv",
]

print(f"Target: `{CATALOG}`.`{SCHEMA}`  (DATA_SOURCE={DATA_SOURCE})")
print(f"Volume: {VOLUME_PATH}")

## Section 2: Create the schema and volume

The volume is where the raw CSVs land before they are loaded into Delta tables.

`CREATE SCHEMA` requires the target **catalog to already exist**. If you have the
privilege to create catalogs and yours does not exist yet, uncomment the `CREATE CATALOG`
line below; otherwise set `CATALOG` in Section 1 to a catalog you can write to (for
example `main`).

In [ ]:
# spark.sql(f"CREATE CATALOG IF NOT EXISTS `{CATALOG}`")  # uncomment if you can create catalogs
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`")
print(f"Schema and volume ready: {VOLUME_PATH}")

## Section 3: Stage the CSVs into the volume

Read each file from the chosen `DATA_SOURCE` and write it into the volume. Staging into
the volume first means the load step reads every file the same way (`read_files`),
regardless of where it came from.

In [ ]:
import shutil
import urllib.request


def stage(filename):
    dest = f"{VOLUME_PATH}/{filename}"
    if DATA_SOURCE == "github":
        with urllib.request.urlopen(f"{GITHUB_DATA_BASE}/{filename}") as response:
            data = response.read()
        with open(dest, "wb") as f:
            f.write(data)
    elif DATA_SOURCE == "local":
        shutil.copyfile(f"{LOCAL_DATA_DIR}/{filename}", dest)
    else:
        raise ValueError(f"Unknown DATA_SOURCE '{DATA_SOURCE}'. Use 'github' or 'local'.")
    print(f"  staged {filename} -> {dest}")


print(f"Staging {len(CSV_FILES)} files (DATA_SOURCE={DATA_SOURCE})...")
for name in CSV_FILES:
    stage(name)
print("Done.")

## Section 4: Create the base tables

`CREATE OR REPLACE TABLE` defines column types and Unity Catalog column comments. The
comments are the primary signal Genie and other tooling use to understand the data, so
they live in the DDL and survive every re-run. The table definitions match
`enrichment-pipeline/sql/schema.sql` in the source repo.

In [ ]:
# table_name -> (column DDL with comments + table comment, read schema, typed SELECT)
TABLES = {
    "accounts": {
        "ddl": """(
            account_id   BIGINT  NOT NULL COMMENT 'Unique account identifier (primary key)',
            account_hash STRING           COMMENT 'Anonymized account identifier derived from the original account number',
            account_type STRING           COMMENT 'Account category: checking, savings, or business',
            region       STRING           COMMENT 'Geographic region where the account was opened',
            balance      DOUBLE           COMMENT 'Current account balance in USD',
            opened_date  DATE             COMMENT 'Date the account was opened',
            holder_age   INT              COMMENT 'Age of the account holder in years',
            CONSTRAINT accounts_pk PRIMARY KEY (account_id) RELY
        )
        USING DELTA
        COMMENT 'Account dimension - one row per account holder'""",
        "read_schema": "account_id STRING, account_hash STRING, account_type STRING, region STRING, balance STRING, opened_date STRING, holder_age STRING",
        "select": """CAST(account_id  AS BIGINT) AS account_id,
            account_hash,
            account_type,
            region,
            CAST(balance     AS DOUBLE) AS balance,
            CAST(opened_date AS DATE)   AS opened_date,
            CAST(holder_age  AS INT)    AS holder_age""",
    },
    "merchants": {
        "ddl": """(
            merchant_id   BIGINT  NOT NULL COMMENT 'Unique merchant identifier (primary key)',
            merchant_name STRING           COMMENT 'Merchant business name',
            category      STRING           COMMENT 'Merchant business category (e.g., retail, food, entertainment)',
            region        STRING           COMMENT 'Geographic region where the merchant operates',
            CONSTRAINT merchants_pk PRIMARY KEY (merchant_id) RELY
        )
        USING DELTA
        COMMENT 'Merchant dimension - one row per merchant'""",
        "read_schema": "merchant_id STRING, merchant_name STRING, category STRING, region STRING",
        "select": """CAST(merchant_id AS BIGINT) AS merchant_id,
            merchant_name,
            category,
            region""",
    },
    "transactions": {
        "ddl": """(
            txn_id        BIGINT    NOT NULL COMMENT 'Unique transaction identifier (primary key)',
            account_id    BIGINT             COMMENT 'Account that initiated the payment (foreign key to accounts.account_id)',
            merchant_id   BIGINT             COMMENT 'Merchant that received the payment (foreign key to merchants.merchant_id)',
            amount        DOUBLE             COMMENT 'Transaction amount in USD',
            txn_timestamp TIMESTAMP          COMMENT 'Timestamp when the transaction occurred',
            txn_hour      INT                COMMENT 'Hour of day (0-23) when the transaction occurred',
            CONSTRAINT transactions_pk PRIMARY KEY (txn_id) RELY
        )
        USING DELTA
        COMMENT 'Transaction fact table - one row per account-to-merchant payment event'""",
        "read_schema": "txn_id STRING, account_id STRING, merchant_id STRING, amount STRING, txn_timestamp STRING, txn_hour STRING",
        "select": """CAST(txn_id        AS BIGINT)    AS txn_id,
            CAST(account_id    AS BIGINT)    AS account_id,
            CAST(merchant_id   AS BIGINT)    AS merchant_id,
            CAST(amount        AS DOUBLE)    AS amount,
            CAST(txn_timestamp AS TIMESTAMP) AS txn_timestamp,
            CAST(txn_hour      AS INT)       AS txn_hour""",
    },
    "account_links": {
        "ddl": """(
            link_id            BIGINT    NOT NULL COMMENT 'Unique transfer event identifier (primary key)',
            src_account_id     BIGINT             COMMENT 'Account that sent the transfer (foreign key to accounts.account_id)',
            dst_account_id     BIGINT             COMMENT 'Account that received the transfer (foreign key to accounts.account_id)',
            amount             DOUBLE             COMMENT 'Transfer amount in USD',
            transfer_timestamp TIMESTAMP          COMMENT 'Timestamp when the transfer occurred',
            CONSTRAINT account_links_pk PRIMARY KEY (link_id) RELY
        )
        USING DELTA
        COMMENT 'Account-to-account transfer graph - one row per directed transfer event'""",
        "read_schema": "link_id STRING, src_account_id STRING, dst_account_id STRING, amount STRING, transfer_timestamp STRING",
        "select": """CAST(link_id            AS BIGINT)    AS link_id,
            CAST(src_account_id     AS BIGINT)    AS src_account_id,
            CAST(dst_account_id     AS BIGINT)    AS dst_account_id,
            CAST(amount             AS DOUBLE)    AS amount,
            CAST(transfer_timestamp AS TIMESTAMP) AS transfer_timestamp""",
    },
    "account_labels": {
        "ddl": """(
            account_id BIGINT  NOT NULL COMMENT 'Account identifier (foreign key to accounts.account_id)',
            is_fraud   BOOLEAN          COMMENT 'Ground-truth fraud label: true if the account is a confirmed fraud ring member',
            CONSTRAINT account_labels_pk PRIMARY KEY (account_id) RELY
        )
        USING DELTA
        COMMENT 'Ground-truth fraud labels - one row per account'""",
        "read_schema": "account_id STRING, is_fraud STRING",
        "select": """CAST(account_id AS BIGINT) AS account_id,
            CAST(CASE WHEN lower(is_fraud) = 'true' THEN 'true' ELSE 'false' END AS BOOLEAN) AS is_fraud""",
    },
}

for name, t in TABLES.items():
    spark.sql(f"CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.{name} {t['ddl']}")
    print(f"  created `{CATALOG}`.`{SCHEMA}`.{name}")

## Section 5: Load the data

`INSERT OVERWRITE` replaces all rows without touching the schema or column comments.
Each table reads its staged CSV with `read_files` (all columns as strings) and casts to
the typed schema, mirroring the original pipeline's load step.

In [ ]:
for name, t in TABLES.items():
    spark.sql(f"""
        INSERT OVERWRITE `{CATALOG}`.`{SCHEMA}`.{name}
        SELECT
            {t['select']}
        FROM read_files(
            '{VOLUME_PATH}/{name}.csv',
            format      => 'csv',
            header      => 'true',
            inferSchema => 'false',
            schema      => '{t['read_schema']}'
        )
    """)
    print(f"  loaded {name}")

## Section 6: Add the foreign key constraints

These informational `FOREIGN KEY ... RELY` constraints are the declared-relationship
signal downstream tooling (and the Virtual Graph schema) read. Unity Catalog does not
enforce them. Each constraint is dropped first if present, so this cell is safe to re-run
on its own.

In [ ]:
FOREIGN_KEYS = [
    ("transactions",   "transactions_account_fk",   "account_id",     "accounts",  "account_id"),
    ("transactions",   "transactions_merchant_fk",  "merchant_id",    "merchants", "merchant_id"),
    ("account_links",  "account_links_src_fk",      "src_account_id", "accounts",  "account_id"),
    ("account_links",  "account_links_dst_fk",      "dst_account_id", "accounts",  "account_id"),
    ("account_labels", "account_labels_account_fk", "account_id",     "accounts",  "account_id"),
]

for table, fk_name, fk_col, ref_table, ref_col in FOREIGN_KEYS:
    spark.sql(f"ALTER TABLE `{CATALOG}`.`{SCHEMA}`.{table} DROP CONSTRAINT IF EXISTS {fk_name}")
    spark.sql(f"""
        ALTER TABLE `{CATALOG}`.`{SCHEMA}`.{table}
        ADD CONSTRAINT {fk_name} FOREIGN KEY ({fk_col})
        REFERENCES `{CATALOG}`.`{SCHEMA}`.{ref_table} ({ref_col}) RELY
    """)
    print(f"  {table}.{fk_col} -> {ref_table}.{ref_col}")

## Section 7: Verify

Count the rows in each table. This is the authoritative check on what landed.

In [ ]:
print("Row counts:")
for name in TABLES:
    count = spark.sql(f"SELECT count(*) AS n FROM `{CATALOG}`.`{SCHEMA}`.{name}").collect()[0]["n"]
    print(f"  {name:<16} {count:>8,}")

## Next steps

The five Silver tables are loaded:

| Table | What it is |
|---|---|
| `accounts` | Account dimension |
| `merchants` | Merchant dimension |
| `transactions` | Account-to-merchant payments |
| `account_links` | Account-to-account transfers |
| `account_labels` | Ground-truth fraud labels |

From here:

1. **Build the Virtual Graph** over these tables in Aura with node labels `:Account` and
   `:Merchant` and the `TRANSACTED_WITH` / `TRANSFERRED_TO` relationships. See
   [`virtual-graph.md`](../virtual-graph.md).
2. **Run the demos** from the project root once your `.env` has the `NEO4J_*` values:

   ```bash
   uv run vg-demo                 # fraud demo (default)
   uv run vg-demo --demo basic    # exploration / visualization queries
   ```